# 01 data

Step 1 of the guide: load Sparkov, strip PII, sort by card and time, split by time (test = fraudTest period, val = last 30 days before it), basic EDA.

Writes `ROOT/data/processed/transactions.parquet` and `ROOT/results/01_data.json`, plus two charts.

In [ ]:
import os, sys
from google.colab import drive, userdata
drive.mount("/content/drive")
ROOT = "/content/drive/MyDrive/Colab Notebooks/tfm"
tok = userdata.get("GH_TOKEN")
if not os.path.exists("/content/transaction-fm"):
    !git clone -q https://{tok}@github.com/lb603487485/transaction-fm.git /content/transaction-fm
else:
    !cd /content/transaction-fm && git pull -q
%pip install -q datasets xgboost
%cd /content/transaction-fm
sys.path.insert(0, "/content/transaction-fm/src")
import tfm
print("tfm ok")

## Run step 1 (about 2 to 3 minutes for 1.85M rows)

In [ ]:
import json
from tfm.data import run
stats = run(raw_dir=f"{ROOT}/data/raw", out_dir=f"{ROOT}/data/processed", results_dir=f"{ROOT}/results")
print(json.dumps({k: v for k, v in stats.items() if k != "fraud_rate_by_month"}, indent=2))

## Charts: fraud rate by month, and the gap between a card's consecutive transactions

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
m = pd.Series(stats["fraud_rate_by_month"])
m.plot(ax=ax[0], marker="o")
ax[0].axvline(list(m.index).index(stats["val_start"][:7]), color="grey", ls="--", label="val start")
ax[0].axvline(list(m.index).index(stats["test_start"][:7]), color="red", ls="--", label="test start")
ax[0].set_title("fraud rate by month"); ax[0].legend(); ax[0].tick_params(axis="x", rotation=45)

df = pd.read_parquet(f"{ROOT}/data/processed/transactions.parquet", columns=["card_id", "ts"])
gaps = df.groupby("card_id")["ts"].diff().dt.total_seconds().div(3600).dropna()
ax[1].hist(gaps.clip(upper=240), bins=60)
ax[1].set_title("hours between consecutive transactions of a card (clipped at 240)")
plt.tight_layout()
plt.savefig(f"{ROOT}/results/01_data_eda.png", dpi=120)
plt.show()

## What to read off the numbers

- `fraud_rate` around 0.5%, and `fraud_rate_per_split` tells you whether test is harder or easier than train.
- `fraud_burst_share`: share of fraud rows whose previous fraud on the same card was within 24 h. High means frauds come in bursts, so card-level discovery and leakage checks matter.
- `gap_hours` percentiles decide the time-gap bins in step 2.
- `n_merchants` (about 700) decides the merchant top-N cutoff in step 2.

## Push results and this notebook back to GitHub

In [ ]:
import shutil, glob, os, json
os.makedirs("results/metrics", exist_ok=True); os.makedirs("results/figures", exist_ok=True)
for f in glob.glob(f"{ROOT}/results/*.json"):
    shutil.copy(f, "results/metrics/")
for f in glob.glob(f"{ROOT}/results/*.png"):
    shutil.copy(f, "results/figures/")
for f in glob.glob(f"{ROOT}/notebooks/*.ipynb"):
    nb = json.load(open(f))
    for c in nb["cells"]:
        if c["cell_type"] == "code":
            c["outputs"] = []; c["execution_count"] = None
    json.dump(nb, open(os.path.join("notebooks", os.path.basename(f)), "w"), indent=1, ensure_ascii=False)
!git config user.email "linboyy@hotmail.com" && git config user.name "Bo Lin"
!git add results notebooks && git commit -q -m "step 1 results" && git push -q
!git log --oneline | head -1

In [ ]:
drive.flush_and_unmount()